# MJX 03 — Control & Inverse Kinematics (SO-101 + XLeRobot)

### Lab Description

This lab moves from *setting joint angles* to *commanding an end-effector pose*. You will drive real robot arms — the **SO-101** (the arm used by Hugging Face LeRobot) and the dual-arm mobile **XLeRobot** — by solving **inverse kinematics (IK)**.

We use **Damped Least Squares (DLS)** IK with MuJoCo's analytic Jacobian (`mj_jacBody`). DLS is the standard, robust IK method: it nudges the joints toward a target end-effector position while staying numerically stable near singularities. The damping term trades a little accuracy for a lot of stability.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Map joint names to their `qpos` / Jacobian-column addresses
- Implement a Damped Least Squares IK step using `mj_jacBody`
- Drive the SO-101 end-effector along a target trajectory
- Run IK on one arm of the XLeRobot mobile manipulator
- Frame robots cleanly with a fitted free camera and the visual geom group

### Import libraries and IK helpers

We define four helpers up front:
- `joint_addrs` — map joint names to their `qpos` indices and Jacobian (dof) columns.
- `dls_ik_step` — one Damped Least Squares update: compute the end-effector error, get the Jacobian with `mj_jacBody`, and solve `dq = Jᵀ(JJᵀ + λ²I)⁻¹ e`.
- `fit_free_camera` — an auto-framing free camera with optional distance / look-at overrides.
- `visual_only_option` — show only the dominant visual geom group (the MJX 02 trick).

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import mujoco
import imageio

os.makedirs("output/videos", exist_ok=True)


def joint_addrs(model, names):
    """Map joint names to their qpos and dof (Jacobian column) addresses."""
    qadr, dadr = [], []
    for n in names:
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n)
        qadr.append(int(model.jnt_qposadr[jid]))
        dadr.append(int(model.jnt_dofadr[jid]))
    return qadr, dadr


def dls_ik_step(model, data, ee_body_id, target_pos, qadr, dadr, damping=0.15, step=0.5):
    """One Damped Least Squares IK update toward a 3D target position."""
    mujoco.mj_forward(model, data)
    err = target_pos - data.xpos[ee_body_id]
    jacp = np.zeros((3, model.nv))
    jacr = np.zeros((3, model.nv))
    mujoco.mj_jacBody(model, data, jacp, jacr, ee_body_id)
    J = jacp[:, dadr]
    dq = J.T @ np.linalg.solve(J @ J.T + (damping ** 2) * np.eye(3), err)
    for k, a in enumerate(qadr):
        data.qpos[a] += step * dq[k]
    return float(np.linalg.norm(err))


def fit_free_camera(model, data, dist=None, elev=-20.0, azim=120.0, lookat=None):
    """A free camera that auto-frames the model, with optional distance/lookat override."""
    cam = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(model, cam)
    if dist is not None:
        cam.distance = dist
    if lookat is not None:
        cam.lookat[:] = lookat
    cam.elevation = elev
    cam.azimuth = azim
    return cam


def visual_only_option(model):
    """Hide collision shapes by showing only the model's visual geom group.

    Some models (e.g. SO-ARM100) put visual meshes in a dedicated non-zero
    group; others (e.g. XLeRobot) keep every geom in group 0. When there is no
    separate visual group we just show all groups."""
    import collections
    opt = mujoco.MjvOption()
    counts = collections.Counter(int(model.geom_group[i]) for i in range(model.ngeom))
    nonzero = [g for g in counts if g != 0]
    if not nonzero:
        opt.geomgroup[:] = 1
        return opt
    visual_grp = max(nonzero, key=lambda g: counts[g])
    opt.geomgroup[:] = 0
    opt.geomgroup[visual_grp] = 1
    return opt

### SO-101 — load the arm

`robot_descriptions` auto-downloads the SO-ARM100 (SO-101) MJCF from the MuJoCo Menagerie. We compile it, pick the arm joints, and use the last body in the kinematic tree (near the jaw) as the end-effector proxy.

In [ ]:
# robot_descriptions auto-downloads the SO-ARM100 MJCF from MuJoCo Menagerie.
from robot_descriptions import so_arm100_mj_description as so_arm

so_model = mujoco.MjModel.from_xml_path(so_arm.MJCF_PATH)
so_data = mujoco.MjData(so_model)
mujoco.mj_forward(so_model, so_data)

ARM_JOINTS = ["Rotation", "Pitch", "Elbow", "Wrist_Pitch", "Wrist_Roll"]
so_qadr, so_dadr = joint_addrs(so_model, ARM_JOINTS)
# End-effector proxy: the last body in the kinematic tree (around the jaw).
so_ee = so_model.nbody - 1
print("SO-101 end-effector body:", mujoco.mj_id2name(so_model, mujoco.mjtObj.mjOBJ_BODY, so_ee))
print("start EE pos:", np.round(so_data.xpos[so_ee], 3))

### SO-101 — follow a square with IK

We trace a small square in front of the arm. For each waypoint we run a few `dls_ik_step` iterations so the end-effector converges, then render a frame. The camera (`azim=-60`) gives a front 3/4 view so the base sits naturally on the ground.

In [ ]:
so_renderer = mujoco.Renderer(so_model, height=320, width=320)
so_opt = visual_only_option(so_model)
so_cam = fit_free_camera(so_model, so_data, dist=0.7, elev=-20, azim=-60)

# A small square trajectory in the vertical plane, centered near the start pose.
center = so_data.xpos[so_ee].copy()
r = 0.08
square = []
for t in np.linspace(0, 1, 80):
    ang = 2 * np.pi * t
    square.append(center + np.array([0.0, r * np.cos(ang), r * np.sin(ang)]))

frames, errs = [], []
for target in square:
    for _ in range(6):  # a few IK iterations per waypoint
        e = dls_ik_step(so_model, so_data, so_ee, target, so_qadr, so_dadr, step=0.6)
    errs.append(e)
    so_renderer.update_scene(so_data, camera=so_cam, scene_option=so_opt)
    frames.append(so_renderer.render())

imageio.mimsave("output/videos/mjx03_so101.mp4", frames, fps=20)
print(f"SO-101: {len(frames)} frames, final tracking error = {errs[-1]*1000:.1f} mm")

### Watch the SO-101 trace the square

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx03_so101.mp4")

### XLeRobot — load the dual-arm mobile manipulator

XLeRobot has a mobile base and two 5-DOF arms. We sparse-clone only its MuJoCo model from the upstream repo and compile it. (`nq`/`nu`/`nbody` show how much bigger this model is than the single SO-101 arm.)

In [ ]:
import subprocess, os

# Sparse-clone only the MuJoCo model directory from the XLeRobot repo.
if not os.path.exists("XLeRobot/simulation/mujoco/scene.xml"):
    subprocess.run(
        "git clone --depth 1 --filter=blob:none --sparse "
        "https://github.com/Vector-Wangel/XLeRobot.git XLeRobot",
        shell=True, check=True,
    )
    subprocess.run(
        "cd XLeRobot && git sparse-checkout set simulation/mujoco",
        shell=True, check=True,
    )

xle_model = mujoco.MjModel.from_xml_path("XLeRobot/simulation/mujoco/scene.xml")
xle_data = mujoco.MjData(xle_model)
mujoco.mj_forward(xle_model, xle_data)
print("XLeRobot: nq", xle_model.nq, "nu", xle_model.nu, "nbody", xle_model.nbody)

### XLeRobot — select the right arm

We grab the right-arm joints (excluding the jaw) and use the wrist-roll link as the end-effector. The base and left arm stay fixed while we move the right arm with IK.

In [ ]:
# Right-arm joints (exclude the jaw/gripper from position IK).
RIGHT_ARM = ["Rotation_R", "Pitch_R", "Elbow_R", "Wrist_Pitch_R", "Wrist_Roll_R"]
xle_qadr, xle_dadr = joint_addrs(xle_model, RIGHT_ARM)

# End-effector proxy: the body of the last right-arm joint (wrist roll link).
wrist_jid = mujoco.mj_name2id(xle_model, mujoco.mjtObj.mjOBJ_JOINT, "Wrist_Roll_R")
xle_ee = int(xle_model.jnt_bodyid[wrist_jid])
print("XLeRobot right-arm EE body:", mujoco.mj_id2name(xle_model, mujoco.mjtObj.mjOBJ_BODY, xle_ee))
print("start EE pos:", np.round(xle_data.xpos[xle_ee], 3))

### XLeRobot — reach a target with IK

We interpolate from the current end-effector position toward a goal offset and run DLS IK each step. The right arm reaches while the left arm and base stay put. We view both arms from the front, with a bit of the cart body in shot.

In [ ]:
xle_renderer = mujoco.Renderer(xle_model, height=360, width=480)
xle_opt = visual_only_option(xle_model)
# Front view of BOTH arms with a bit of the cart body in shot. The two arms are
# mounted on top of the blue cart (grippers near z~0.9, facing +y), so we look
# from the front (azim=90) at arm height and pull back enough to catch the cart.
xle_cam = fit_free_camera(xle_model, xle_data, dist=1.0, elev=-12, azim=90,
                          lookat=np.array([0.0, 0.0, 0.88]))

# Reach toward a target offset from the right-arm start pose.
start = xle_data.xpos[xle_ee].copy()
goal = start + np.array([0.10, -0.05, 0.12])

frames, errs = [], []
for t in np.linspace(0, 1, 90):
    target = start + t * (goal - start)
    e = dls_ik_step(xle_model, xle_data, xle_ee, target, xle_qadr, xle_dadr, step=0.4)
    errs.append(e)
    xle_renderer.update_scene(xle_data, camera=xle_cam, scene_option=xle_opt)
    frames.append(xle_renderer.render())

imageio.mimsave("output/videos/mjx03_xlerobot.mp4", frames, fps=20)
print(f"XLeRobot: {len(frames)} frames, final tracking error = {errs[-1]*1000:.1f} mm")

### Watch the XLeRobot reach

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx03_xlerobot.mp4")

## Conclusions

You implemented Damped Least Squares IK and drove two real robot arms to follow Cartesian targets using MuJoCo's analytic Jacobian. This closes out **Module A** (pure-MuJoCo foundations). Module B (MJX 04–05) re-implements this physics in JAX so we can run thousands of simulations in parallel on the GPU.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT